# Part 12 · Notebook 08 — Recovery, backups and security

**Sessions:** S15–S16 (Reliability, disaster recovery & chaos testing · Security & operational risk) · [Lesson plan](../../docs/lessons/PART_12_TRADING_PLATFORM_CAPSTONE.md) · graded labs in [`labs/part12/`](../../labs/part12/)

**You will:**
1. Rebuild the platform's state from its fill journal, idempotently, on top of an atomic snapshot.
2. Crash the engine at many points and recover; trade again only after a clean reconciliation.
3. Verify a backup's checksum before restoring it.
4. Scan for secrets, rank threats, and add broker-side limits as a second line of defense.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
Everything runs offline on synthetic data with known answers (planted violations, injected faults, a strategy that stops working): no broker, Docker or network is needed.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p12lib.py is in notebooks/part12/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p12lib as p

p.use_course_style()

## 1. State is a function of events

The engine's state (positions, cash) is never edited in place: it is rebuilt from the **fill journal**, one fill at a time (`p.apply_fill`). Each fill has a sequence number, and the state remembers the last one applied, so replaying a fill that the state already holds must do nothing. That makes recovery safe: replay generously, and duplicates cannot double-count.

In [ ]:
rng = np.random.default_rng(0)
journal = [{"seq": i, "symbol": str(rng.choice(["SPY", "QQQ", "IWM"])), "qty": int(rng.choice([-100, 100])),
            "price": round(float(rng.uniform(95, 105)), 2)} for i in range(40)]
journal[:3]

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def rebuild(events, state=None):
    state = state or {"positions": {}, "cash": 0.0, "last_seq": -1}
    for f in events:
        ...                                                          # ✍️ apply (p.apply_fill) only fills newer than state["last_seq"]
    return state

cases = {"from scratch": (journal,), "snapshot at 25, replay from 20": (journal[20:], p.rebuild(journal[:26])),
         "journal delivered twice": (journal + journal,)}
mine = p.check("rebuild", {k: p.attempt(rebuild, *a) for k, a in cases.items()}, {k: p.rebuild(*a) for k, a in cases.items()})
mine["from scratch"]

## 2. Crash and recover

The engine writes each fill to the journal first, and snapshots its state every 5 fills with `p.save_snapshot` (write to a temporary file, then rename: a crash mid-write never leaves half a snapshot). `p.recover` loads the snapshot, replays the journal on top, and reconciles with the broker; trading resumes only if nothing breaks. Kill the engine at ten different points:

In [ ]:
import tempfile

work = Path(tempfile.mkdtemp())
rows = []
for crash in range(3, 40, 4):                                       # the engine dies right after fill `crash`
    snap = work / f"state_{crash}.json"
    last_snap = crash // 5 * 5 - 1                                   # snapshots after seq 4, 9, 14, …
    if last_snap >= 0:
        p.save_snapshot(p.rebuild(journal[: last_snap + 1]), snap)
    written = journal[: crash + 1]                                   # the journal survived the crash
    broker = p.rebuild(written)["positions"]                         # the broker's truth
    out = p.recover(snap, written, broker)
    rows.append({"crash after seq": crash, "snapshot at seq": last_snap if last_snap >= 0 else "none",
                 "state correct": out["state"] == p.rebuild(written), "may trade": out["may_trade"]})
pd.DataFrame(rows).set_index("crash after seq").T

Correct at every crash point, including the one before the first snapshot. Now someone places a manual order in TWS while the engine is down:

In [ ]:
broker = dict(p.rebuild(journal)["positions"])
broker["IWM"] = broker.get("IWM", 0) + 100                           # a manual buy, placed outside the platform
out = p.recover(work / "state_39.json", journal, broker)
print("breaks:", out["breaks"], "| may trade:", out["may_trade"])
print("open orders:", p.reconcile_orders({"O101", "O102"}, {"O102", "TWS-5531"}))

The platform refuses to trade until a human explains the break. That is the rule: the broker is the truth, and an unexplained difference means our model of the world is wrong.

**Backups** are only real if they restore. Store the checksum apart from the file, and verify it *before* restoring:

In [ ]:
tables = {"positions": p.rebuild(journal)["positions"], "orders": [{"id": "O101", "status": "filled"}], "date": "2025-06-02"}
blob, digest = p.backup(tables)
print(f"backup: {len(blob)} bytes, sha256 {digest[:16]}…; restored equal: {p.restore(blob, digest) == tables}")
corrupt = blob[:10] + bytes([blob[10] ^ 1]) + blob[11:]              # one flipped bit on the disk
try:
    p.restore(corrupt, digest)
except ValueError as err:
    print("restoring the damaged copy:", err)

Recovery targets: the **RTO** (how long until trading again) and the **RPO** (how much data you may lose). The journal plus snapshots keep the RPO at zero fills; the drill measures the RTO. Chaos tests (kill a container, drop the network, fill the disk, skew the clock) run in paper, and every failure gets a one-page runbook.

## 3. Security

Secrets in code end up in Git history, backups and screenshots. A scanner runs in CI over every file, line by line: report `(file, line number from 1, rule name)` for every `p.SECRET_RULES` pattern that matches a line, sorted.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
import re

def scan_secrets(files):
    out = []
    for name, text in files.items():
        for i, line in enumerate(text.splitlines(), start=1):
            ...                                                      # ✍️ for each rule, pattern in p.SECRET_RULES: record a match
    return sorted(out)

FILES = {"quantforge/config.py": 'BROKER_HOST = "127.0.0.1"\npassword = "Tr4d1ng!2025"\n',
         "deploy/secrets.example.env": "DB_PASSWORD=${DB_PASSWORD}\nANTHROPIC_API_KEY=${ANTHROPIC_API_KEY}\n",
         "research/scratch.py": 'client = anthropic.Anthropic(api_key="' + "sk-ant-" + "api03-" + "x" * 24 + '")\n',
         "quantforge/apps/api.py": 'token = request.headers["X-Token"]\npassword = os.environ["API_PASSWORD"]\n'}
p.check("scan_secrets", p.attempt(scan_secrets, FILES), p.scan_secrets(FILES))

The hard-coded password and the pasted API key are found; `${...}` references and values read from the environment are not. A found key is **rotated**, not just deleted: it is still in the history.

A **threat model** lists what can go wrong, how likely (1–5) and how bad (1–5); risk = likelihood × impact, and the top three get fixed first:

In [ ]:
threats = pd.DataFrame([("bad data feed", 4, 4, "RobustZ on ticks, a second feed to cross-check"),
                        ("runaway strategy", 3, 5, "order-rate limit, kill switch"),
                        ("stolen API key", 2, 5, "IP allow-list, 2FA, rotation, separate paper/live keys"),
                        ("VPS down at the open", 3, 3, "restart policies, runbook, broker-side stops"),
                        ("gateway stuck at login", 4, 2, "IBC daily restart, pre-market check"),
                        ("clock drift", 2, 2, "chrony, alert on skew")],
                       columns=["threat", "likelihood", "impact", "control"])
p.rank_threats(threats)

The platform's risk engine is the first line of defense. The second is at the broker (IB's precautionary settings): limits on order size, notional and distance from the last price, which hold even if the platform has a bug. Return the failed checks in the order `size`, `notional`, `price_band` (limit price more than `band_pct` from the last price).

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def precautionary_check(order, last_price, limits):
    out = []
    if abs(order["qty"]) > limits["max_qty"]:
        out.append("size")
    ...                                                              # ✍️ "notional" (|qty| · price), then "price_band"
    return out

LIMITS = {"max_qty": 2_000, "max_notional": 250_000, "band_pct": 0.03}
orders = {"normal": {"qty": 300, "price": 512.10}, "large but plausible": {"qty": 600, "price": 512.10},
          "fat finger: 30,000 shares": {"qty": 30_000, "price": 512.10}, "price typo: 51.21": {"qty": 300, "price": 51.21},
          "both": {"qty": 30_000, "price": 5121.0}}
mine = p.check("precautionary_check", {k: p.attempt(precautionary_check, o, 512.0, LIMITS) for k, o in orders.items()},
               {k: p.precautionary_check(o, 512.0, LIMITS) for k, o in orders.items()})
mine

## Wrap-up

* State is rebuilt from the journal; replay is idempotent; snapshots are atomic.
* Recovery ends with a reconciliation, and trading waits for a clean one.
* A backup is verified before it is restored, and restores are tested.
* No secrets in code; rank threats by likelihood × impact; broker-side limits behind the risk engine.
* Graded version: `labs/part12/week44_production` (`apply_fill`, `rebuild`, `save_snapshot`, `recover`, `backup`, `restore`, `scan_secrets`, `rank_threats`, `precautionary_check`).